In [1]:
import sys
sys.path.append('/Users/aleksei/projects/code-of-kutulu-client')

In [2]:
!pwd

/Users/aleksei/projects/code-of-kutulu-client/notebooks


In [3]:
# import torch

In [4]:
from tqdm import tqdm
from collections import Counter
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
from datetime import datetime
import pickle as pkl
import zlib
import base64
import torch

/Users/aleksei/.local/share/virtualenvs/kutulu-_n6nfavE/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:

from src.envs.trainer import Trainer, WOOD_MAZES, BRONZE_MAZES
from src.game.template import DEFAULT_KUTULU_ACTIONS, EXTENDED_KUTULU_ACTIONS

In [6]:
def get_score(result):
    agent_id = 0
    metrics_list = result[2]
    acc = [
        [
            metrics[key][agent_id][0]
            for key in [
                'check_exp_normal', 'check_exp_coridor', 'check_exp_corner',
                'check_wan_normal', 'check_wan_coridor', 'check_wan_corner',
            ]
        ] for metrics in metrics_list[-10:]
    ]
    top_action_cnt = [
        [
            metrics[key][agent_id][2]
            for key in [
                'check_exp_normal',
                'check_wan_normal',
            ]
        ] for metrics in metrics_list[-10:]
    ]
    mean_q = [
        [
            metrics[key][agent_id][4]
            for key in [
                'check_exp_normal',
                'check_wan_normal',
            ]
        ] for metrics in metrics_list[-10:]
    ]
    score = np.mean(acc) + 0.25 * np.mean(top_action_cnt) - 0.1 * np.max(np.max(mean_q, axis=0) - np.min(mean_q, axis=0))
    return score

In [11]:
def get_trainer(
    gamma, size,
    fc_dim, conv_dim, 
    random_epsilon,
    sanity_coef, reward_for_win, reward_for_lose,
    entropy_coef, value_loss_coef,
    clip_ratio,
    ppo_epochs, mini_batch_size,
    target_kl, max_grad_norm,
    gae_lambda, n_step,
):
    env_kwargs = {
        'reward_params': {
            'sanity_coef': sanity_coef, 'reward_for_win': reward_for_win, 'reward_for_lose': reward_for_lose
        }
    }
    random_agent_info = {
        'train': False,
        'type': 'epsilon_wait',
        'action_space_n': ACTION_SPACE_N,
        'epsilon_params': {'start': random_epsilon, 'final': random_epsilon, 'decay': int(4 * 10**5)},
        'state_type': 'closest',
        'action': 'WAIT',
    }
    research_agent_info = {
        'train': True,
        'type': 'ppo',
        'action_space_n': ACTION_SPACE_N,
        'state_type': 'conv',
        'model_params': {
            'fc_dim': fc_dim,
            'conv_dim': conv_dim,
            'size': size,
        },
        'gamma': gamma,
        'lr': 1e-4,
        'optimizer': 'adamw',
        'scheduler_params': {'type': 'cosine', 'T_max': 800},
        'entropy_coef': entropy_coef,
        'value_loss_coef': value_loss_coef,
        'clip_ratio': clip_ratio,
        'ppo_epochs': ppo_epochs,
        'mini_batch_size': mini_batch_size,
        'target_kl': target_kl,
        'max_grad_norm': max_grad_norm,
        'gae_lambda': gae_lambda,
        'n_step': n_step,
    }
    agents_info = [research_agent_info]
    for i in range(len(agents_info), 4):
        agents_info.append(dict(random_agent_info))
    assert len(agents_info) == 4
    trainer = Trainer(
        num_experiments=NUM_EXPERIMENTS, agents_info=agents_info, shuffle=True,
        league_level=LEAGUE_LEVEL, mazes=MAZES, actions=ACTIONS, log_dir='../runs', verbose=False,
        env_kwargs=env_kwargs, silent=True,
    )
    return trainer

In [12]:
LEAGUE_LEVEL = 3

MAZES = BRONZE_MAZES if LEAGUE_LEVEL >= 3 else WOOD_MAZES
ACTIONS = EXTENDED_KUTULU_ACTIONS if LEAGUE_LEVEL >= 3 else DEFAULT_KUTULU_ACTIONS
ACTION_SPACE_N = len(ACTIONS)
NUM_EXPERIMENTS = 1000

In [13]:
research_info_list = [{
    'gamma': 0.9092072223045214,
    'size': 4,
    'fc_dim': 32,
    'conv_dim': 16,
    'random_epsilon': 0.9210248619457186,
    'reward_for_win': None,
    'reward_for_lose': -2,
    'sanity_coef': 0.40779196741184814,
    'entropy_coef': 0.01,
    'value_loss_coef': 0.5,
    'clip_ratio': 0.2,
    'ppo_epochs': 4,
    'mini_batch_size': 64,
    'target_kl': 0.01,
    'max_grad_norm': 0.5,
    'gae_lambda': 0.95,
    'n_step': 10,
}]

In [18]:
for research_info in research_info_list:
    trainer = get_trainer(**research_info)
    print(research_info)
    result = trainer.train()

{'gamma': 0.9092072223045214, 'size': 4, 'fc_dim': 32, 'conv_dim': 16, 'random_epsilon': 0.9210248619457186, 'reward_for_win': None, 'reward_for_lose': -2, 'sanity_coef': 0.40779196741184814, 'entropy_coef': 0.01, 'value_loss_coef': 0.5, 'clip_ratio': 0.2, 'ppo_epochs': 4, 'mini_batch_size': 64, 'target_kl': 0.01, 'max_grad_norm': 0.5, 'gae_lambda': 0.95, 'n_step': 10}


100%|██████████| 1000/1000 [05:51<00:00,  2.85it/s]


In [26]:
get_score(result)

0.7362605447873648

In [14]:
for research_info in research_info_list:
    trainer = get_trainer(**research_info)
    break

In [15]:
trainer.verbose = True
trainer.shuffle = True
trainer.only_train = True

In [17]:
# trainer.play_rollout()

In [14]:
trainer.agent_map

array([3, 0, 1, 2])